In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, chi2_contingency

df = pd.read_csv("../data/processed/telco_clean.csv")
print(df.shape)

(7043, 21)


In [2]:
rows = []
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    churned = df.loc[df["Churn"] == 1, col]
    stayed = df.loc[df["Churn"] == 0, col]
    u, p = mannwhitneyu(churned, stayed)
    rows.append({
        "feature": col,
        "median_churned": churned.median(),
        "median_stayed": stayed.median(),
        "p_value": p,
        "prob_churner_higher": round(u / (len(churned) * len(stayed)), 3),
    })
pd.DataFrame(rows)

,feature,median_churned,median_stayed,p_value,prob_churner_higher
0,tenure,10.00,38.000,2.419636e-208,0.260
1,MonthlyCharges,79.65,64.425,3.311628e-54,0.621
2,TotalCharges,703.55,1679.525,5.685034e-83,0.350


In [3]:
cat_cols = list(df.select_dtypes(include="object").columns.drop("customerID"))
cat_cols.append("SeniorCitizen")  # stored as 0/1, but it is really a category

results = []
for col in cat_cols:
    table = pd.crosstab(df[col], df["Churn"])
    chi2, p, _, _ = chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    results.append({"feature": col, "cramers_v": round(v, 3), "p_value": p})

ranking = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
ranking

,feature,cramers_v,p_value
12,Contract,0.410,5.863038e-258
6,OnlineSecurity,0.347,2.661150e-185
9,TechSupport,0.343,1.443084e-180
5,InternetService,0.322,9.571788e-160
14,PaymentMethod,0.303,3.682355e-140
7,OnlineBackup,0.292,2.079759e-131
8,DeviceProtection,0.282,5.505219e-122
11,StreamingMovies,0.231,2.667757e-82
10,StreamingTV,0.231,5.528994e-82
13,PaperlessBilling,0.191,4.073355e-58


In [4]:
n_tests = len(ranking) + 3        # categorical tests + 3 numeric tests
threshold = 0.05 / n_tests
print("Bonferroni threshold:", threshold)
ranking["significant"] = ranking["p_value"] < threshold
ranking[["feature", "cramers_v", "p_value", "significant"]]

Bonferroni threshold: 0.002631578947368421


,feature,cramers_v,p_value,significant
12,Contract,0.410,5.863038e-258,True
6,OnlineSecurity,0.347,2.661150e-185,True
9,TechSupport,0.343,1.443084e-180,True
5,InternetService,0.322,9.571788e-160,True
14,PaymentMethod,0.303,3.682355e-140,True
7,OnlineBackup,0.292,2.079759e-131,True
8,DeviceProtection,0.282,5.505219e-122,True
11,StreamingMovies,0.231,2.667757e-82,True
10,StreamingTV,0.231,5.528994e-82,True
13,PaperlessBilling,0.191,4.073355e-58,True


In [5]:
pd.DataFrame(rows).query("feature == 'tenure'")
pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index").mul(100).round(2)
pd.DataFrame(rows).query("feature == 'MonthlyCharges'")
ranking.sort_values("cramers_v")[["feature", "cramers_v", "p_value", "significant"]]

,feature,cramers_v,p_value,significant
0,gender,0.008,4.865787e-01,False
3,PhoneService,0.011,3.387825e-01,False
4,MultipleLines,0.040,3.464383e-03,False
1,Partner,0.150,2.139911e-36,True
15,SeniorCitizen,0.150,1.510067e-36,True
2,Dependents,0.164,4.924922e-43,True
13,PaperlessBilling,0.191,4.073355e-58,True
11,StreamingMovies,0.231,2.667757e-82,True
10,StreamingTV,0.231,5.528994e-82,True
8,DeviceProtection,0.282,5.505219e-122,True


In [6]:
# CHURNLENS - FINAL EDA FINDINGS

from scipy.stats import mannwhitneyu

print("\n========== NUMERICAL FEATURES ==========")

for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    churned = df.loc[df["Churn"] == 1, col]
    stayed = df.loc[df["Churn"] == 0, col]

    u, p = mannwhitneyu(churned, stayed)

    print(f"\n{col}")
    print("Churned median:", round(churned.median(), 2))
    print("Stayed median:", round(stayed.median(), 2))
    print("P-value:", f"{p:.4e}")
    print("Probability churner has higher value:", round(
        (u + 0.5 * ((len(churned) * len(stayed)) -
        (2*u))) / (len(churned) * len(stayed)), 3
    ))

print("\n========== CATEGORICAL FEATURES ==========")

for col in ["Contract", "PaymentMethod"]:
    print(f"\n{col} - Churn percentages")
    print(
        pd.crosstab(df[col], df["Churn"], normalize="index")
        [1].mul(100).round(2).to_string()
    )

print("\n========== WEAK ASSOCIATIONS ==========")
print(
    ranking.sort_values("cramers_v")
    [["feature", "cramers_v", "p_value", "significant"]]
    .head(5).to_string(index=False)
)


========== NUMERICAL FEATURES ==========

tenure
Churned median: 10.0
Stayed median: 38.0
P-value: 2.4196e-208
Probability churner has higher value: 0.5

MonthlyCharges
Churned median: 79.65
Stayed median: 64.43
P-value: 3.3116e-54
Probability churner has higher value: 0.5

TotalCharges
Churned median: 703.55
Stayed median: 1679.52
P-value: 5.6850e-83
Probability churner has higher value: 0.5

========== CATEGORICAL FEATURES ==========

Contract - Churn percentages
Contract
Month-to-month    42.71
One year          11.27
Two year           2.83

PaymentMethod - Churn percentages
PaymentMethod
Bank transfer (automatic)    16.71
Credit card (automatic)      15.24
Electronic check             45.29
Mailed check                 19.11

========== WEAK ASSOCIATIONS ==========
      feature  cramers_v      p_value  significant
       gender      0.008 4.865787e-01        False
 PhoneService      0.011 3.387825e-01        False
MultipleLines      0.040 3.464383e-03        False
      Partner 